# 08 Render Target Resume

Stage 8 renders the targeted resume JSON into human-readable resume documents.

This stage does not generate new resume content. It loads the assembled target resume artifact from Stage 7 and passes it through the existing rendering system.

Inputs:

* `artifacts/target_resume_v1.json`
* `layouts/standard_v6.yaml`
* `renderer_v6.py`

Outputs:

* Rendered DOCX resume
* Rendered PDF resume, if supported by the rendering workflow

Stage 8 is intentionally deterministic. The targeted resume JSON is the source of truth for content, while the layout file controls presentation. If content needs to change, update the upstream artifacts rather than editing the rendered document directly.

Key checks:

* Confirm all expected sections render.
* Confirm section ordering matches the target resume JSON.
* Confirm bullets, subsections, dates, and headings display correctly.
* Confirm the rendered resume is readable and appropriately concise.
* Capture any rendering or layout issues separately from content issues.


In [1]:
from pathlib import Path

from src.config import ARTIFACT_DIR
from src.helpers import load_json, save_json

In [2]:
target_resume = load_json(ARTIFACT_DIR / "target_resume_v1.json")

[(idx, section.get("heading"), section.get("type")) for idx, section in enumerate(target_resume["sections"])]

[(0, 'Summary', 'paragraph'),
 (1, 'CORE EXPERTISE', 'bullet'),
 (2, 'Professional Experience', 'experience'),
 (3, 'Education', 'subsections'),
 (4, 'Patents & Recognition', 'bullet'),
 (5, 'Selected Projects', 'subsections')]

## Rendering Boundary

This notebook should not rewrite resume content.

Minor layout fixes belong here. Content fixes should go back to the appropriate upstream stage:

* Summary or Core Expertise issues: Stage 6
* Experience bullet issues: Stage 7
* Evidence selection issues: Stage 4
* Source-of-truth factual issues: canonical resume


In [3]:
from pathlib import Path

from src.config import ARTIFACT_DIR
from src.helpers import load_json, load_yaml
from src.renderer import render_resume, convert_docx_to_pdf

TARGET_RESUME_PATH = ARTIFACT_DIR / "target_resume_v1.json"
TRACE_PATH = ARTIFACT_DIR / "target_resume_trace_v1.json"

LAYOUT_PATH = Path("layouts/standard_v6.yaml")
RENDERER_PATH = Path("src/renderer.py")

OUTPUT_DOCX_PATH = ARTIFACT_DIR / "target_resume_v1.docx"
OUTPUT_PDF_PATH = ARTIFACT_DIR / "target_resume_v1.pdf"

In [4]:
target_resume = load_json(TARGET_RESUME_PATH)
layout = load_yaml(LAYOUT_PATH)

[(idx, section.get("heading"), section.get("type")) for idx, section in enumerate(target_resume["sections"])]

[(0, 'Summary', 'paragraph'),
 (1, 'CORE EXPERTISE', 'bullet'),
 (2, 'Professional Experience', 'experience'),
 (3, 'Education', 'subsections'),
 (4, 'Patents & Recognition', 'bullet'),
 (5, 'Selected Projects', 'subsections')]

In [5]:
def find_renderer_schema_issues(target_resume_or_section):
    """
    Check for renderer schema issues before saving/rendering.

    Accepts either the full target resume or just the Professional Experience section.
    """
    if target_resume_or_section.get("type") == "experience":
        sections = [target_resume_or_section]
    else:
        sections = target_resume_or_section.get("sections", [])

    issues = []

    for section_idx, section in enumerate(sections):
        if section.get("type") != "experience":
            continue

        for exp_idx, exp in enumerate(section.get("content", [])):
            exp_type = exp.get("type")
            content = exp.get("content", [])

            if exp_type == "subsections":
                for item_idx, item in enumerate(content):
                    if not isinstance(item, dict):
                        issues.append({
                            "section_idx": section_idx,
                            "experience_idx": exp_idx,
                            "item_idx": item_idx,
                            "role": exp.get("role"),
                            "organization": exp.get("organization"),
                            "issue": "subsections content item is not a dict",
                            "bad_item_type": type(item).__name__,
                        })

            elif exp_type == "bullet":
                for item_idx, item in enumerate(content):
                    if not isinstance(item, str):
                        issues.append({
                            "section_idx": section_idx,
                            "experience_idx": exp_idx,
                            "item_idx": item_idx,
                            "role": exp.get("role"),
                            "organization": exp.get("organization"),
                            "issue": "bullet content item is not a string",
                            "bad_item_type": type(item).__name__,
                        })

            else:
                issues.append({
                    "section_idx": section_idx,
                    "experience_idx": exp_idx,
                    "role": exp.get("role"),
                    "organization": exp.get("organization"),
                    "issue": f"unexpected experience entry type: {exp_type}",
                })

    return issues


In [6]:
target_resume = load_json(TARGET_RESUME_PATH)

schema_issues = find_renderer_schema_issues(target_resume)
schema_issues

[]

In [7]:
if schema_issues:
    raise ValueError(f"Renderer schema issues found: {schema_issues}")

render_resume(
    resume=target_resume,
    layout=layout,
    output_path=OUTPUT_DOCX_PATH,
)

OUTPUT_DOCX_PATH, OUTPUT_DOCX_PATH.exists()

(PosixPath('artifacts/target_resume_v1.docx'), True)

In [8]:
try:
    pdf_path = convert_docx_to_pdf(OUTPUT_DOCX_PATH)

    print("PDF created:", pdf_path)
    print("DOCX exists:", OUTPUT_DOCX_PATH.exists())
    print("PDF exists:", pdf_path.exists())

except Exception as e:
    print("PDF conversion failed.")
    print(type(e).__name__, str(e))
    print(
        "DOCX rendering succeeded, so this is likely a local LibreOffice / soffice issue, "
        "not a resume pipeline issue."
    )

convert /Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/target_resume_v1.docx as a Writer document -> /Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/target_resume_v1.pdf using filter : writer_pdf_Export
Overwriting: /Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/target_resume_v1.pdf
PDF created: /Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/target_resume_v1.pdf
DOCX exists: True
PDF exists: True


In [9]:
rendered_outputs = {
    "docx": OUTPUT_DOCX_PATH.exists(),
    "pdf": OUTPUT_PDF_PATH.exists(),
}

rendered_outputs

{'docx': True, 'pdf': True}

## Stage 8 Completion Summary

Stage 8 renders the targeted resume JSON into document outputs.

Outputs:

- `artifacts/target_resume_v1.docx`
- `artifacts/target_resume_v1.pdf`, if PDF conversion succeeds

Key decisions:

- Resume content comes from `target_resume_v1.json`.
- Presentation comes from `layouts/standard_v6.yaml`.
- Rendering logic comes from `src/renderer.py`.
- Stage 8 validates and renders; it does not repair or rewrite resume content.
- If content issues are found, fix the upstream artifact and rerender.

Stage 8 is complete when:

- The DOCX renders successfully.
- The PDF conversion succeeds, or any failure is confirmed as a local LibreOffice / `soffice` issue.
- The expected resume sections appear in the correct order.
- The rendered document is visually acceptable for first-pass review.

## Lessons Learned

This project showed that targeted resume generation works best when content, evidence, positioning, and rendering are treated as separate concerns.

A few major lessons stood out:

1. **Define contracts before generation.**
   The pipeline became much more reliable once each stage had explicit inputs, outputs, and artifact names. Instead of asking an LLM to “write a resume,” each stage performed a smaller job: extract evidence, map evidence to market signals, score evidence, select evidence, infer capabilities, generate positioning, assemble the resume, and render the final document.

2. **Separate candidate truth from market interpretation.**
   The canonical resume is the source of truth. Market signals and target archetypes should influence selection and framing, but they should not rewrite the underlying facts. This helped prevent the model from inventing or exaggerating claims.

3. **Score evidence, not bullets.**
   Resume bullets are presentation. Evidence is the underlying accomplishment. Scoring the evidence first made it easier to decide what deserved resume space before worrying about wording.

4. **LLMs are useful for semantic grouping, but not final authority.**
   The LLM performed well when asked to group evidence into capability themes or draft positioning language. Human review was still necessary for naming, tone, emphasis, and overstatement control.

5. **Human-in-the-loop patches are a feature, not a bug.**
   Manual naming and positioning patches improved the resume. The key was preserving traceability back to capabilities and evidence IDs so human judgment did not break auditability.

6. **Traceability reduces resume drift.**
   Keeping separate trace artifacts made it easier to understand why each bullet, capability, and Core Expertise line existed. This is especially important when LLMs generate polished language.

7. **Rendering should be deterministic.**
   Once the target resume JSON exists, rendering should not rewrite content. Stage 8 worked best when it validated schema compatibility and rendered the document without repairing or changing content.

8. **Schema discipline matters.**
   The renderer issue with raw strings inside `subsections` showed why each stage should emit renderer-safe JSON. Small schema mismatches can break downstream automation even when the content itself is good.

9. **Versioned artifacts make iteration safer.**
   Outputs like `target_resume_v1.json`, `target_resume_trace_v1.json`, and rendered DOCX/PDF files created clear checkpoints. This made it easier to debug without losing earlier work.

10. **The final system is more valuable than a single resume.**
    The completed pipeline can generate a targeted resume while preserving evidence discipline. The broader value is a reusable framework for turning career history into market-specific resume artifacts.


## Opportunities to Improve

The current pipeline works end-to-end, but several improvements would make it more reliable, reusable, and easier to maintain. These are ordered by expected return on investment.

### 1. Add formal schema validation between stages

Highest ROI improvement by far.

Several issues came from schema assumptions, especially around `experience`, `subsections`, and `bullet` content. Add lightweight validation after each artifact is created.

Recommended artifacts to validate:

* `candidate_evidence.json`
* `signal_evidence_map_v1.json`
* `evidence_scores.json`
* `selected_evidence.json`
* `capability_review.json`
* `resume_positioning.json`
* `target_resume_v1.json`

This would catch downstream failures earlier and make notebook execution safer.

### 2. Move candidate-specific rules into configuration

Some rules are intentionally candidate-specific, such as caution rules for teaching evidence, older evidence, NBCUniversal short-cycle work, and roadmap/assessment language.

These should eventually move from notebook code into a candidate configuration file, such as:

* `config/candidate_profile.yaml`
* `config/caution_rules.yaml`
* `config/resume_positioning_preferences.yaml`

This would make the pipeline more reusable without pretending the current project is fully generalized.

### 3. Add an artifact manifest

Create a single manifest that records each stage, input artifacts, output artifacts, timestamp, model used, and notes.

Example:

```json
{
  "stage": "06_resume_positioning",
  "inputs": [
    "capability_review.json",
    "selected_evidence.json",
    "target_archetype.json"
  ],
  "outputs": [
    "resume_positioning.json"
  ],
  "model": "gpt-4.1",
  "notes": "Manual positioning patch applied with evidence traceability preserved."
}
```

This would make the project easier to audit and explain.

### 4. Improve experience bullet review

Stage 7 generates targeted experience bullets, but this is the highest-risk content generation step. Add a review table showing each generated bullet, source role, supporting evidence IDs, cautions, and original evidence text.

This would make it easier to catch overstatement before rendering.

### 5. Add resume length and density controls

The current renderer produces a document, but the pipeline does not yet reason about length, density, or target page count.

Useful controls:

* Maximum bullets per role
* Maximum total bullets
* Optional inclusion of Selected Projects
* One-page vs two-page mode
* Compact vs expanded Core Expertise

This should come after schema validation and bullet review, because layout optimization is only useful once content is stable.

### 6. Support multiple target resume variants

The pipeline currently produces one target resume. It could support variants such as:

* Principal AI Architect
* AI Platform / MLOps Lead
* GenAI Solutions Architect
* Data Science / Analytics Leadership
* Technical Instructor / AI Enablement

The existing artifact structure already supports this. The main addition would be target-specific output folders or filenames.

### 7. Add comparison views across versions

As the pipeline improves, it would help to compare:

* `target_resume_v1.json` vs `target_resume_v2.json`
* generated bullets vs original evidence
* Core Expertise versions
* rendered output length and section counts

This is lower ROI than schema validation, but useful once multiple versions exist.

### 8. Add automated rendering checks

After DOCX/PDF creation, add checks for:

* Expected section headings
* Empty sections
* Unexpected unsupported block types
* Missing Education or Patents & Recognition
* Number of experience entries
* Number of bullets per role

This would prevent silent rendering issues.

### 9. Improve market signal deduplication

The evidence scoring stage can overcount overlapping market signals. The field-capped score helped, but future work could deduplicate semantically similar signals before scoring.

This would make evidence scores more stable and interpretable.

### 10. Package common notebook functions into modules

Several helper functions are now mature enough to move into `src/`, especially:

* schema validation
* evidence grouping
* section copying
* display helpers
* renderer preflight checks

This should be done gradually. Avoid over-packaging while the workflow is still evolving.
